In [1]:
import importlib.util
import subprocess
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), Path.cwd() / 'LRAEP-WR-Trainer') if (p / 'src' / 'LRAEP_trainer').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
try:
    get_ipython().history_manager.enabled = False
    get_ipython().history_manager.db_log_output = False
except NameError:
    pass


In [2]:
modules = {'numpy': 'numpy', 'torch': 'torch', 'blake3': 'blake3', 'pydantic': 'pydantic', 'loguru': 'loguru', 'cryptography': 'cryptography', 'ipykernel': 'ipykernel', 'Pillow': 'PIL', 'pyarrow': 'pyarrow'}
missing = [line for line in (ROOT / 'requirements.txt').read_text().splitlines() if importlib.util.find_spec(modules[line.split('>=')[0]]) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
print('Ready')


Ready


In [3]:
import logging
import os

if 'trainer' in globals() and hasattr(trainer, 'stop_training'):
    trainer.stop_training()
    del trainer

def run_training(root):
    root = Path(root).resolve()
    os.environ.setdefault('LRAEP_CACHE_DIR', str(root / '.runtime/cache'))
    os.environ.setdefault('LRAEP_DATA_DIR', str(root / '.runtime'))
    logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s', stream=sys.stdout, force=True)
    try:
        import LRAEP_trainer
        if Path(LRAEP_trainer.__file__).resolve().parent != (root / 'src/LRAEP_trainer').resolve():
            raise RuntimeError('Restart the kernel before using this source checkout')
        from LRAEP_trainer.config import load_config, resolve_pool_settings
        from LRAEP_trainer.network import routed
        from LRAEP_trainer.pool_training import run_pool_training, validate_pool_endpoint, validate_pool_identity
        from LRAEP_trainer.vpn import ManagedVPN
        config = load_config(root / 'config.json')
        if config['training']['job_sequence'][0][2] == -1:
            return 0
        config['pool'] = resolve_pool_settings(config['pool'])
        if config['pool']['protocol'] != 'WR':
            raise ValueError('WR is required')
        if not config['vpn']['enabled'] or not config['vpn']['link']:
            raise ValueError('An enabled VPN link is required')
        validate_pool_endpoint(config['pool'])
        validate_pool_identity(config['pool'], validate_address=False)
        with ManagedVPN(config['vpn']) as proxy, routed(proxy):
            print('WR training | VPN enabled', flush=True)
            return run_pool_training(config)
    except KeyboardInterrupt:
        print('Training interrupted.', flush=True)
        return 130
    except (ValueError, RuntimeError, ImportError, OSError) as error:
        print(f'Training failed ({type(error).__name__}): {error}', flush=True)
        return 1

print('Training started. Interrupt this cell to stop.', flush=True)
exit_code = run_training(ROOT)
print(f'Training stopped (exit code {exit_code}).', flush=True)


Training started. Interrupt this cell to stop.
WR training | VPN enabled
2026-10-30 15:35:19,519 GPU: NVIDIA GeForce RTX 3060
2026-10-30 15:35:19,522 Mining backend: native CUDA DP4A


Hashrate(avg) 0.00 H/s | GPU N/A | accepted=0 rejected=0 | submitted=0 batches=0 attempts=0 stale=0


2026-10-30 15:35:19,523 Side-job effort GPU duty-cycle target: 79%


Side-job | Epoch 1 | Preparing | GPU N/A


2026-10-30 15:35:21,321 Resumed side-job weights at epoch 3
2026-10-30 15:35:21,322 Side-job CNN: NVIDIA GeForce RTX 3060; 11,440,194 parameters
2026-10-30 15:35:29,581 Side-job epoch 4 (164 steps, 5248 examples, 32 passes)


...

2026-10-30 15:35:44,462 Side-job effort GPU duty-cycle target: 65%


Side-job | Epoch 4 | Step 19/164 (11.6%) | loss=0.8593 accuracy=51.8% | GPU 78%
Side-job | Epoch 4 | Step 21/164 (12.8%) | loss=0.8416 accuracy=52.5% | GPU 60%
Side-job | Epoch 4 | Step 23/164 (14.0%) | loss=0.8266 accuracy=53.1% | GPU 50%
Side-job | Epoch 4 | Step 25/164 (15.2%) | loss=0.8154 accuracy=53.2% | GPU 60%


2026-10-30 15:35:49,786 Side-job effort GPU duty-cycle target: 77%


Side-job | Epoch 4 | Step 27/164 (16.5%) | loss=0.8055 accuracy=53.1% | GPU 65%
Side-job | Epoch 4 | Step 28/164 (17.1%) | loss=0.7995 accuracy=53.7% | GPU 76%


2026-10-30 15:35:50,788 Pool training stopped: batches=0 submitted=0 accepted=0 rejected=0 stale=0
Training stopped (exit code 0).
